# 面试问题：BatchNorm、LayerNorm、RMSNorm 有什么区别，如何不用现成归一化层手写？

**一句话回答**：BatchNorm 对 batch/空间维按 channel 统计，训练依赖同批样本并维护 running mean/var，推理使用冻结统计；LayerNorm 对每个样本最后若干特征做均值/方差，不依赖 batch；RMSNorm 只除以均方根、不减均值。选择取决于张量语义、小 batch、序列长度、分布式同步和残差位置。

本 Notebook 只用 PyTorch 基础张量算子实现三个 `nn.Module` 与 `forward`，验证 train/eval、梯度、低精度和 Pre-Norm 残差。

In [ ]:
import hashlib, json, math
import numpy as np
import torch
from torch import nn

SEED95=9501; torch.manual_seed(SEED95); np.random.seed(SEED95)
x95=torch.randn(8,6)*3+4
assert x95.shape==(8,6) and torch.isfinite(x95).all()
assert SEED95==9501
assert torch.__version__

## 1. 手写二维 BatchNorm

输入 `[N,C]`，训练期每 channel 在 N 上计算 biased batch variance 用于归一化；running variance 更新常用 unbiased estimate。输出 `gamma*xhat+beta`。推理不能重新用线上 batch 统计，否则单条请求和 batch 组成会改变结果。

本例显式更新 buffer，running state 会随 batch 顺序变化。

In [ ]:
class BatchNorm95(nn.Module):
    def __init__(self,features,eps=1e-5,momentum=.1):
        super().__init__(); self.weight=nn.Parameter(torch.ones(features)); self.bias=nn.Parameter(torch.zeros(features)); self.eps=eps; self.momentum=momentum; self.register_buffer("running_mean",torch.zeros(features)); self.register_buffer("running_var",torch.ones(features)); self.register_buffer("batches",torch.tensor(0,dtype=torch.long))
    def forward(self,x):
        if x.ndim!=2 or x.shape[1]!=self.weight.numel(): raise ValueError("bn_shape_contract")
        if self.training:
            mean=x.float().mean(0); var=x.float().var(0,unbiased=False)
            with torch.no_grad():
                unbiased=x.float().var(0,unbiased=True) if x.shape[0]>1 else var; self.running_mean.lerp_(mean,self.momentum); self.running_var.lerp_(unbiased,self.momentum); self.batches+=1
        else: mean=self.running_mean; var=self.running_var
        y=(x.float()-mean)/torch.sqrt(var+self.eps); return (y*self.weight+self.bias).to(x.dtype)
bn95=BatchNorm95(6); ybn95=bn95(x95)
assert torch.allclose(ybn95.mean(0),torch.zeros(6),atol=1e-5)
assert torch.allclose(ybn95.var(0,unbiased=False),torch.ones(6),atol=2e-5)
assert bn95.batches.item()==1 and not torch.allclose(bn95.running_mean,torch.zeros(6))

## 2. BatchNorm train/eval 与 batch dependence

train 模式下同一个样本与不同同伴组成 batch，输出会变化；eval 使用 running stats，同一样本输出稳定。小 batch 的统计噪声大，batch=1 时 variance 为 0，训练输出几乎只剩 beta。

分布式训练若每卡 batch 小，可用 SyncBN 聚合统计，但增加通信；这与梯度 all-reduce 是不同 collective。

In [ ]:
sample95=x95[:1]; batch_a95=torch.cat([sample95,torch.randn(7,6)],0); batch_b95=torch.cat([sample95,torch.randn(7,6)+10],0); train_a95=bn95(batch_a95)[0]; train_b95=bn95(batch_b95)[0]
assert not torch.allclose(train_a95,train_b95)
bn95.eval(); eval_a95=bn95(batch_a95)[0]; eval_b95=bn95(batch_b95)[0]
assert torch.allclose(eval_a95,eval_b95)
single_bn95=BatchNorm95(6); single95=single_bn95(sample95)
assert torch.allclose(single95,torch.zeros_like(single95),atol=1e-4)

## 3. 手写 LayerNorm

对每个 token/sample 的最后 `D` 维求 mean/variance，batch 中其他样本不参与。Transformer 输入 `[B,T,D]` 时通常按 D 归一化，每个 token 独立；padding token 虽会被归一化，但后续 attention/loss 仍必须 mask。

gamma/beta 形状等于 normalized shape，支持广播。

In [ ]:
class LayerNorm95(nn.Module):
    def __init__(self,features,eps=1e-5): super().__init__(); self.weight=nn.Parameter(torch.ones(features)); self.bias=nn.Parameter(torch.zeros(features)); self.eps=eps
    def forward(self,x):
        if x.shape[-1]!=self.weight.numel(): raise ValueError("ln_shape_contract")
        xf=x.float(); mean=xf.mean(-1,keepdim=True); var=xf.var(-1,unbiased=False,keepdim=True); return (((xf-mean)/torch.sqrt(var+self.eps))*self.weight+self.bias).to(x.dtype)
ln95=LayerNorm95(6); yln95=ln95(x95)
assert torch.allclose(yln95.mean(-1),torch.zeros(8),atol=1e-5)
assert torch.allclose(yln95.var(-1,unbiased=False),torch.ones(8),atol=2e-5)
assert torch.allclose(ln95(batch_a95)[0],ln95(batch_b95)[0])

## 4. 手写 RMSNorm

`RMS(x)=sqrt(mean(x²)+eps)`，输出 `gamma*x/RMS(x)`；不减均值，通常也无 bias，计算更简单。缩放输入 `a*x`（a>0）后输出近似不变，但平移输入会改变输出。LayerNorm 对平移与正缩放都近似不变。

RMSNorm 不保证输出均值为 0。

In [ ]:
class RMSNorm95(nn.Module):
    def __init__(self,features,eps=1e-6): super().__init__(); self.weight=nn.Parameter(torch.ones(features)); self.eps=eps
    def forward(self,x):
        if x.shape[-1]!=self.weight.numel(): raise ValueError("rms_shape_contract")
        xf=x.float(); rms=torch.sqrt(xf.square().mean(-1,keepdim=True)+self.eps); return (xf/rms*self.weight).to(x.dtype)
rms95=RMSNorm95(6); yrms95=rms95(x95)
assert torch.allclose(yrms95.square().mean(-1),torch.ones(8),atol=1e-5)
assert not torch.allclose(yrms95.mean(-1),torch.zeros(8),atol=1e-2)
assert torch.allclose(rms95(x95),rms95(x95*3),atol=2e-6) and not torch.allclose(rms95(x95),rms95(x95+3))

## 5. 归一化轴与不变性对比

BatchNorm 的 channel 统计跨样本，适合卷积大 batch；Layer/RMS 沿 hidden 维，适合变长序列和自回归单样本推理。错误的 axis 会让模型看似能跑但语义改变，例如把 `[B,T,D]` 在 T 上归一化会混合序列长度。

下面用置换验证 LN/RMS 对 batch 排列逐样本等价，BN running state 则依赖训练历史。

In [ ]:
perm95=torch.tensor([3,1,7,0,6,2,5,4]); inv95=torch.argsort(perm95)
assert torch.allclose(ln95(x95[perm95])[inv95],ln95(x95))
assert torch.allclose(rms95(x95[perm95])[inv95],rms95(x95))
assert ln95.weight.shape==rms95.weight.shape==torch.Size([6]) and bn95.running_mean.shape==torch.Size([6])

## 6. Pre-Norm 与 Post-Norm

Pre-Norm block：`x + F(Norm(x))`，残差有直接恒等梯度路径，深 Transformer 通常更易优化；Post-Norm：`Norm(x+F(x))`。两者不是简单移动一行代码，训练动态和最终表示尺度不同。

用同一线性层构造两个 block，检查形状与梯度有限；这里只展示结构，不声称一个小样本能决定架构优劣。

In [ ]:
class PreNormBlock95(nn.Module):
    def __init__(self,d): super().__init__(); self.norm=RMSNorm95(d); self.ff=nn.Linear(d,d,bias=False)
    def forward(self,x): return x+torch.tanh(self.ff(self.norm(x)))
class PostNormBlock95(nn.Module):
    def __init__(self,d): super().__init__(); self.norm=LayerNorm95(d); self.ff=nn.Linear(d,d,bias=False)
    def forward(self,x): return self.norm(x+torch.tanh(self.ff(x)))
xr95=torch.randn(4,6,requires_grad=True); pre95=PreNormBlock95(6); post95=PostNormBlock95(6); loss95=pre95(xr95).square().mean()+post95(xr95).square().mean(); loss95.backward()
assert pre95(xr95).shape==post95(xr95).shape==xr95.shape
assert torch.isfinite(xr95.grad).all() and xr95.grad.norm()>0
assert all(p.grad is not None for p in list(pre95.parameters())+list(post95.parameters()))

## 7. epsilon、低精度与状态发布

mean/variance/RMS 通常用 FP32 accumulation，再 cast 回输入 dtype。epsilon 太小可能在常数输入/FP16 下放大噪声，太大会改变尺度。BatchNorm 发布必须包含 running buffers 和 `eval()` 状态；漏掉 buffer 只加载 gamma/beta 会严重漂移。

running stats 属于模型 state，不是可随请求更新的 cache。

In [ ]:
constant95=torch.full((4,6),1000.,dtype=torch.float16); ln_fp95=LayerNorm95(6).half(); rms_fp95=RMSNorm95(6).half(); out_ln_fp95=ln_fp95(constant95); out_rms_fp95=rms_fp95(constant95)
assert torch.isfinite(out_ln_fp95).all() and torch.isfinite(out_rms_fp95).all()
assert torch.allclose(out_ln_fp95,torch.zeros_like(out_ln_fp95),atol=1e-3)
assert torch.allclose(out_rms_fp95,torch.ones_like(out_rms_fp95),atol=1e-3)

## 8. Correctness oracle 与 manifest

测试包括：目标轴均值/方差、batch 置换、BN train/eval、单样本、scale/shift 不变性、梯度、constant/FP16、state_dict round-trip。manifest 绑定 norm 类型、axis、eps、momentum、affine 和 running state 摘要。

替换 Norm 会改变 checkpoint 参数与优化动态，不能只改配置继续无验证训练。

In [ ]:
state_bytes95=b"".join(v.detach().cpu().numpy().tobytes() for _,v in sorted(bn95.state_dict().items())); manifest95={"schema":1,"batch_norm":{"features":6,"eps":bn95.eps,"momentum":bn95.momentum,"state_sha256":hashlib.sha256(state_bytes95).hexdigest()},"layer_norm":{"axis":-1,"eps":ln95.eps},"rms_norm":{"axis":-1,"eps":rms95.eps}}
digest95=hashlib.sha256(json.dumps(manifest95,sort_keys=True,separators=(",",":")).encode()).hexdigest()
assert len(digest95)==64 and len(manifest95["batch_norm"]["state_sha256"])==64
assert manifest95["layer_norm"]["axis"]==manifest95["rms_norm"]["axis"]==-1
assert bn95.training is False
print({"bn_batches":int(bn95.batches),"ln_mean_abs":float(yln95.mean(-1).abs().max()),"rms_energy":float(yrms95.square().mean())})

## 9. 面试收束、参考与练习

回答闭环：统计轴 → BN running state/train-eval → LN 每样本中心化 → RMS 仅能量缩放 → batch/shift 不变性 → Pre/Post-Norm → FP32 stats/state 发布。不要只回答“BN 用 batch，LN 用 layer”。

练习：扩展 BN 到 `[N,C,H,W]`；实现 GroupNorm；比较 SyncBN 通信；验证 state_dict 重载；构造 padding 在错误轴污染统计的反例。

参考：[Batch Normalization](https://arxiv.org/abs/1502.03167)、[Layer Normalization](https://arxiv.org/abs/1607.06450)、[RMSNorm](https://arxiv.org/abs/1910.07467)。